# Boston 311: Data Preparation and Routing Catalog

This notebook prepares the 2023 and 2024 Boston 311 records for
demand forecasting, crew planning, and the resident intake assistant.

We focus on four departments:
- PWDx: Public Works
- BTDT: Transportation
- PARK: Parks & Recreation
- ISD: Inspectional Services

The notebook checks data quality, aggregates monthly request counts,
and builds a catalog of historically observed routing combinations.

Raw CSV files are stored locally in `data/raw/`.
Generated datasets are saved in `data/processed/`.

In [18]:
from pathlib import Path
import pandas as pd

# Locate the repository from either its root or notebooks folder.
current = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [current, *current.parents]
     if (p / "data" / "raw").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find the project's data/raw folder.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project folder:", PROJECT_ROOT)
print("\nCSV files:")
for path in sorted(RAW_DIR.glob("*.csv")):
    print(path.name)

Project folder: /Users/nithyaa/Documents/Hackathon/boston311-crew-planner

CSV files:
lagan_311_open_data_2023_rvsd3.csv
lagan_311_open_data_2024_rvsd3.csv


In [ ]:
for year in [2023, 2024]:
    path = RAW_DIR / f"lagan_311_open_data_{year}_rvsd3.csv"
    sample = pd.read_csv(path, nrows=5)

    print(f"\n{year} columns:")
    print(sample.columns.tolist())
    display(sample.head(2))

## Load Data and Check Quality

We load the fields needed for this project and retain the source year.
We then filter to the four departments and check for duplicate records,
missing request IDs, and missing values.

Date checks identify parsing failures, opening-year mismatches,
and closure or target dates that occur before opening.
Missing SLA targets and closure dates remain missing.

In [ ]:
columns_needed = [
    "case_enquiry_id",
    "open_dt",
    "sla_target_dt",
    "closed_dt",
    "on_time",
    "case_status",
    "case_title",
    "subject",
    "reason",
    "type",
    "department",
    "queue",
    "source",
    "neighborhood",
]

frames = []

for year in [2023, 2024]:
    path = RAW_DIR / f"lagan_311_open_data_{year}_rvsd3.csv"

    frame = pd.read_csv(
        path,
        usecols=columns_needed,
        dtype="string",
    )

    frame["source_year"] = year
    frames.append(frame)

    print(f"{year}: {len(frame):,} rows")

raw_requests = pd.concat(frames, ignore_index=True)

# Remove surrounding spaces without changing department codes.
raw_requests["department"] = (
    raw_requests["department"].str.strip()
)

display(
    pd.crosstab(
        raw_requests["department"],
        raw_requests["source_year"],
        dropna=False,
    )
)

In [ ]:
DEPARTMENTS = {
    "PWDx": "Public Works",
    "BTDT": "Transportation",
    "PARK": "Parks & Recreation",
    "ISD": "Inspectional Services",
}

requests = raw_requests[
    raw_requests["department"].isin(DEPARTMENTS)
].copy()

# Standardize request IDs before checking duplicates.
requests["case_enquiry_id"] = (
    requests["case_enquiry_id"]
    .str.strip()
    .replace("", pd.NA)
)

print(f"Rows in scope: {len(requests):,}")
print(f"Exact duplicate rows: {requests.duplicated().sum():,}")
print(f"Missing request IDs: {requests['case_enquiry_id'].isna().sum():,}")

repeated_ids = (
    requests["case_enquiry_id"].notna()
    & requests["case_enquiry_id"].duplicated(keep=False)
)

print(f"Rows with repeated request IDs: {repeated_ids.sum():,}")

display(
    requests[
        ["open_dt", "sla_target_dt", "closed_dt", "type", "queue"]
    ]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

if repeated_ids.any():
    display(
        requests.loc[repeated_ids]
        .sort_values("case_enquiry_id")
        .head(10)
    )

In [ ]:
date_columns = ["open_dt", "sla_target_dt", "closed_dt"]
date_checks = []

for column in date_columns:
    original = requests[column].str.strip().replace("", pd.NA)

    parsed = pd.to_datetime(
        original,
        format="mixed",
        errors="coerce",
    )

    date_checks.append({
        "column": column,
        "missing_before": original.isna().sum(),
        "parsing_failures": (original.notna() & parsed.isna()).sum(),
    })

    requests[column] = parsed

display(pd.DataFrame(date_checks))

# Check that opening dates match their source files.
year_mismatch = (
    requests["open_dt"].notna()
    & requests["open_dt"].dt.year.ne(requests["source_year"])
)

print("Opening year mismatches:", year_mismatch.sum())

print(
    "Closed before opened:",
    (requests["closed_dt"] < requests["open_dt"]).sum(),
)

print(
    "Target before opened:",
    (requests["sla_target_dt"] < requests["open_dt"]).sum(),
)

display(
    requests.groupby("source_year")["open_dt"]
    .agg(["min", "max"])
)

All date checks passed: no parsing failures, year mismatches, or dates before opening. We can now create the monthly demand table.

## Create Monthly Demand

Each request contributes one count to its opening month and department.
These counts represent incoming demand, rather than completed work.

The expected output contains 96 rows:
four departments × twelve months × two years.

The forecasting notebook uses `monthly_demand.csv` as its input.

In [ ]:
# Count requests by opening month.
requests["month"] = (
    requests["open_dt"].dt.to_period("M").dt.to_timestamp()
)

monthly_demand = (
    requests
    .groupby(["department", "month"])
    .size()
    .reset_index(name="request_count")
    .sort_values(["department", "month"])
    .reset_index(drop=True)
)

# Validate coverage and totals.
expected_index = pd.MultiIndex.from_product(
    [
        list(DEPARTMENTS),
        pd.date_range("2023-01-01", "2024-12-01", freq="MS"),
    ],
    names=["department", "month"],
)

actual_index = pd.MultiIndex.from_frame(
    monthly_demand[["department", "month"]]
)

missing_months = expected_index.difference(actual_index)

assert len(missing_months) == 0, f"Missing months: {missing_months.tolist()}"
assert len(monthly_demand) == 96
assert monthly_demand["request_count"].sum() == len(requests)

print("Validated: 96 department-month rows.")
display(monthly_demand.head(12))

In [ ]:
#small summary for  forecasting

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "monthly_demand.csv"

monthly_demand.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d",
)

print("Saved:", output_path)

## Build and Explore the Routing Catalog

We count observed combinations of request type, department, and queue.

A request type can have multiple historical routes. We inspect street
cleaning across years and neighborhoods to understand this variation.
Historical frequencies describe past assignments; they do not prove
that a route is correct for a new message.

In [ ]:
# extract the real request type–department–queue combinations for the intake assistant
routing_catalog_full = (
    requests
    .groupby(["type", "department", "queue"])
    .size()
    .reset_index(name="historical_request_count")
    .sort_values(
        ["department", "historical_request_count"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

# Show the 10 most frequent routes per department.
display(
    routing_catalog_full
    .groupby("department", sort=False)
    .head(10)
)

print("Distinct request types:", requests["type"].nunique())
print("Distinct routing combinations:", len(routing_catalog_full))

In [ ]:
# Check which request types have multiple routing combinations
route_counts = (
    routing_catalog_full
    .groupby("type")
    .size()
    .reset_index(name="route_count")
)

display(
    route_counts[route_counts["route_count"] > 1]
    .sort_values("route_count", ascending=False)
)

Some request types have many observed department–queue combinations. That means type alone is insufficient to select a queue reliably. These counts may reflect district-specific queues, transfers, or inconsistent labels—we need to inspect them before deciding.

In [ ]:
# Requests for Street Cleaning
street_cleaning_routes = routing_catalog_full[
    routing_catalog_full["type"] == "Requests for Street Cleaning"
].sort_values("historical_request_count", ascending=False)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
):
    display(street_cleaning_routes)

In [ ]:
#heck how its routes differ between years:
street_cleaning_by_year = (
    requests[
        requests["type"] == "Requests for Street Cleaning"
    ]
    .groupby(["department", "queue", "source_year"])
    .size()
    .unstack("source_year", fill_value=0)
)

street_cleaning_by_year["total"] = (
    street_cleaning_by_year.sum(axis=1)
)

display(
    street_cleaning_by_year
    .sort_values("total", ascending=False)
    .head(20)
)

In [ ]:
# Compare neighborhoods 
street_cleaning = requests[
    requests["type"] == "Requests for Street Cleaning"
].copy()

neighborhood_routes = (
    street_cleaning
    .groupby(
        ["neighborhood", "department", "queue"],
        dropna=False,
    )
    .size()
    .reset_index(name="request_count")
)

# Show the three most frequent routes in each neighborhood.
top_neighborhood_routes = (
    neighborhood_routes
    .sort_values(
        ["neighborhood", "request_count"],
        ascending=[True, False],
    )
    .groupby("neighborhood", dropna=False)
    .head(3)
)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_colwidth", None,
):
    display(top_neighborhood_routes)

In [ ]:
# Keep catalog rows whose type has exactly one routing combination.
routes_per_type = (
    routing_catalog_full
    .groupby("type")["type"]
    .transform("size")
)

simple_types = (
    routing_catalog_full.loc[routes_per_type == 1]
    .copy()
    .sort_values(
        ["department", "historical_request_count"],
        ascending=[True, False],
    )
)

print("Request types with one observed route:", len(simple_types))

with pd.option_context("display.max_colwidth", None):
    display(
        simple_types
        .groupby("department")
        .head(5)
    )

In [ ]:
type_summary = (
    routing_catalog_full
    .groupby(["department", "type"])
    .agg(
        historical_request_count=("historical_request_count", "sum"),
        queue_count=("queue", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["department", "historical_request_count"],
        ascending=[True, False],
    )
)

with pd.option_context(
    "display.max_rows", 40,
    "display.max_colwidth", None,
):
    display(
        type_summary
        .groupby("department")
        .head(8)
        .reset_index(drop=True)
    )

## Select Request Types for the Intake Demo

We limit the intake demonstration to 16 request types to keep the
classification task manageable within the project timeline.

For these types, we retain all observed routes in the selected catalog.
The leading route is the most frequently observed department–queue
combination for a type.

In [ ]:
selected_types = [
    "Parking Enforcement",
    "Sign Repair",
    "Abandoned Vehicles",
    "Traffic Signal Inspection",
    "Rodent Activity",
    "Unsatisfactory Living Conditions",
    "Pest Infestation - Residential",
    "Heat - Excessive Insufficient",
    "Tree Maintenance Requests",
    "Ground Maintenance",
    "Equipment Repair",
    "New Tree Requests",
    "Mattress_Pickup",
    "Schedule a Bulk Item Pickup",
    "Missed Trash/Recycling/Yard Waste/Bulk Item",
    "Request for Pothole Repair",
]

normalized_types = (
    routing_catalog_full["type"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

selected_routes = routing_catalog_full[
    normalized_types.isin(selected_types)
].copy()

matched_types = set(
    normalized_types[normalized_types.isin(selected_types)]
)

assert not (set(selected_types) - matched_types), (
    f"Unmatched types: {set(selected_types) - matched_types}"
)

# Inspect the three most frequent routes per type.
route_preview = (
    selected_routes
    .sort_values(
        ["type", "historical_request_count"],
        ascending=[True, False],
    )
    .groupby("type")
    .head(3)
)

with pd.option_context(
    "display.max_rows", 60,
    "display.max_colwidth", None,
):
    display(route_preview.reset_index(drop=True))
    

In [ ]:
# Work on a copy and add a consistent label for grouping.
catalog = selected_routes.copy()

catalog["type_label"] = (
    catalog["type"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Calculate each route's share of all selected records for its type.
type_totals = (
    catalog.groupby("type_label")["historical_request_count"]
    .transform("sum")
)

catalog["historical_route_share"] = (
    catalog["historical_request_count"] / type_totals
)

catalog = catalog.sort_values(
    ["type_label", "historical_request_count", "department", "queue"],
    ascending=[True, False, True, True],
).reset_index(drop=True)

# Keep every route, not just the top-three preview.
catalog.to_csv(
    PROCESSED_DIR / "routing_catalog_selected.csv",
    index=False,
)

# Review the leading route and its share for each type.
routing_review = (
    catalog.drop_duplicates("type_label")
    [[
        "type_label",
        "department",
        "queue",
        "historical_request_count",
        "historical_route_share",
    ]]
    .rename(columns={
        "department": "leading_department",
        "queue": "leading_queue",
    })
    .copy()
)

routing_review["historical_route_share"] = (
    routing_review["historical_route_share"].mul(100).round(1)
)

# Round percentages only in a display copy.
# Preserve the original unrounded shares for policy decisions.
display_review = routing_review.copy()

display_review["historical_route_share"] = (
    display_review["historical_route_share"].mul(100).round(1)
)

with pd.option_context("display.max_colwidth", None):
    display(display_review.reset_index(drop=True))

## Apply the Historical Routing Review Policy

We compare each leading route's unrounded historical share with 90%.
Rounding is used only when displaying percentages.

- Share at least 90%: historical suggestion.
- Share below 90%: context review required.

This threshold is a team-defined assumption, not an official Boston
rule or an AI confidence score. A high share does not establish
routing correctness.

This notebook records the policy in `routing_review.csv`.
Whether the intake assistant enforces it depends on that notebook's
implementation.

In [ ]:
# Shares remain on a 0–1 scale.
# Apply the 90% threshold before any display rounding.
routing_review["routing_policy"] = (
    routing_review["historical_route_share"]
    .ge(0.90)
    .map({
        True: "historical_suggestion",
        False: "context_review_required",
    })
)

routing_review.to_csv(
    PROCESSED_DIR / "routing_review.csv",
    index=False,
)

print("Saved routing_review.csv")